# Module 1: Renting vs Owning Your Inference

In Module 0 you connected to a GPU you control and confirmed a vLLM endpoint answers. So why operate your own inference server at all, when a hosted API is one HTTP call away? This module makes the tradeoff concrete. You send the same prompt to your self-hosted [vLLM](https://docs.vllm.ai) and, if you have a key, to a hosted API, then look at what actually differs: where your data goes, what each token costs, and who decides when you get throttled. The running example for the whole workshop is the inference server underneath your agents, served on an [Akamai Cloud GPU](https://www.linode.com/products/gpu/).

## Learning objectives
- Send the same prompt to a self-hosted vLLM and a hosted API and compare them
- Read the `usage` object and see the token counts a provider would bill
- Estimate your monthly hosted cost at your own request volume
- Name the three things owning buys you that a price tag hides: data residency, rate-limit control, and tuning control
- Decide when renting is the right call and when owning is

## Prerequisites
- Finished Module 0, with a working endpoint and resolved settings
- A self-hosted vLLM endpoint in `VLLM_HOST`, set in Module 0
- Optional: a hosted OpenAI-compatible key in `HOSTED_API_KEY` to run the comparison live
- About 7 minutes

References: [vLLM](https://docs.vllm.ai) &middot; [OpenAI chat API](https://platform.openai.com/docs/api-reference/chat) &middot; [Akamai Cloud GPUs](https://www.linode.com/products/gpu/) &middot; [Akamai Cloud pricing](https://www.linode.com/pricing/)

## Rent versus own design basics

Inference is the same operation either way: a prompt goes in, tokens come out. The difference is the path and who owns it.

- **Rent.** Your request leaves your network and runs on a provider's shared GPUs. You pay per token, and the provider sets your rate limit.
- **Own.** Your request stays inside your cluster and runs on a GPU only you use. The marginal cost per token is your own hardware, not a price list, and the only ceiling is the card you provisioned.

Neither is wrong. Renting is the right call at low volume or early in a project. The point of this module is to see, with your own numbers, where the line moves.

![Two paths for the same prompt: rent on a provider's GPUs with data leaving your network, or own a vLLM on your GPU with data staying in your cluster](images/01_renting_vs_owning_architecture.png)

## 1. Setup

This workshop reads its connection details from `common/config.py`, which loads them from your environment. Install the one dependency this module needs. We reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q "openai>=1.40"

## 2. Configure endpoint and model

`get_settings()` reads `VLLM_HOST`, `MODEL_NAME`, and `VLLM_API_KEY` from your environment (Module 0 set them). `build_client()` returns an OpenAI client pointed at your vLLM. Pointing `base_url` at your own server is the only change from calling a hosted API, which is the whole convenience of an OpenAI-compatible server.

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.abspath(".."))

from common.config import get_settings, build_client

settings = get_settings()
client = build_client(settings)

print(f"Model:    {settings.model_name}")
print(f"Endpoint: {settings.vllm_host}")

## 3. Call the server you own

Send a prompt to your vLLM endpoint and time it. This request never leaves your cluster. The response carries a `usage` object: the prompt and completion token counts that a hosted provider would put on your bill. On your own server they are free at the margin.

In [ ]:
# Send one prompt to the server you control and measure wall-clock latency.
prompt = "Explain what an LLM inference server does, in two sentences."

start = time.time()
resp = client.chat.completions.create(
    model=settings.model_name,
    messages=[{"role": "user", "content": prompt}],
    max_tokens=128,
    temperature=0.0,
)
elapsed = time.time() - start

print(resp.choices[0].message.content)
print(f"\nself-hosted latency: {elapsed:.2f}s")
print("usage:", resp.usage)   # prompt_tokens + completion_tokens = what a provider would bill

**What you should see:** a two-sentence answer, a latency in the low seconds, and a `usage` object with token counts. Those counts are the unit of a hosted bill. Here they cost you nothing per request.

## 4. Call a hosted API (optional)

If you have a hosted OpenAI-compatible key, run this to compare. The code is nearly identical. The differences are the `base_url` (the provider's, not yours), the model, and the fact that the request leaves your network. Set `HOSTED_API_KEY` first, or skip this cell. The lesson lands either way.

In [ ]:
# Optional. Set HOSTED_API_KEY to run the rented path live; otherwise this skips.
from openai import OpenAI

hosted_key = os.environ.get("HOSTED_API_KEY")
if not hosted_key:
    print("No HOSTED_API_KEY set. Skipping the hosted call. The comparison still applies.")
else:
    hosted = OpenAI(
        base_url=os.environ.get("HOSTED_BASE_URL", "https://api.openai.com/v1"),
        api_key=hosted_key,
    )
    hosted_model = os.environ.get("HOSTED_MODEL", "gpt-4o-mini")
    start = time.time()
    hresp = hosted.chat.completions.create(
        model=hosted_model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=128,
        temperature=0.0,
    )
    print(hresp.choices[0].message.content)
    print(f"\nhosted latency: {time.time() - start:.2f}s   (this prompt left your network)")
    print("usage:", hresp.usage)

**What you should see:** either a skip message, or a similar answer with its own latency and usage. The text may differ because the models differ. The lesson is not which model is smarter. It is who controls the server.

## 5. The bill renting sends you

Hosted inference is priced per token: predictable at low volume, brutal at scale. Put in your own request volume and a provider's price, and watch the monthly number. This is the cost a single-call demo never shows you.

In [ ]:
# Estimate the monthly hosted bill at YOUR volume. Edit these to match your workload.
requests_per_day = 200_000
input_tokens_per_request = 500
output_tokens_per_request = 300

# Hosted prices in dollars per million tokens. Replace with your provider's.
hosted_input_price = 0.15
hosted_output_price = 0.60

in_tokens = requests_per_day * input_tokens_per_request * 30
out_tokens = requests_per_day * output_tokens_per_request * 30
monthly = (in_tokens / 1e6) * hosted_input_price + (out_tokens / 1e6) * hosted_output_price

print(f"monthly input tokens : {in_tokens/1e9:.1f}B")
print(f"monthly output tokens: {out_tokens/1e9:.1f}B")
print(f"estimated hosted bill: ${monthly:,.0f} / month")
print("\nA dedicated GPU is a fixed monthly cost, no matter how many tokens flow through it.")
print("Past some volume, owning is cheaper. Module 8 turns measured throughput into a real cost per million tokens.")

**What you should see:** a monthly token volume and a dollar figure. At 200k requests a day this is real money, and every token you add raises it. A dedicated GPU does not move with token count. That is the crossover this workshop is built to find.

## 6. What a price tag does not show

Cost is the obvious axis. Three others matter as much in production:

- **Data residency.** Your self-hosted request never left your cluster. For regulated data, customer PII, or anything under a contractual boundary, that is the difference between compliant and not. The hosted call sent your prompt to a third party.
- **Rate limits.** A hosted provider sets your throughput and can throttle you during a launch, exactly when you need it most. On your own server the only ceiling is the GPU you provisioned, and you can watch it coming in the metrics (Modules 3 and 4).
- **Control.** You pick the model, the quantization, the context length, and the batching policy. You tune for your traffic instead of accepting one-size-fits-all defaults. That is the rest of this workshop.

## Things to know

- **The model is not the lesson.** A hosted model may answer better or worse than your self-hosted one. This module is about who owns the server, not which model wins.
- **Tokens are the unit of the bill.** The `usage` object on every response is what a provider meters. Watching it is the first habit of running your own inference.
- **vLLM speaks the OpenAI API.** The only change from a hosted call is `base_url`. Your application code, SDKs, and agent frameworks do not care that the server is yours.

> NOTE: The default prices above are illustrative. Use your provider's current per-million-token rates for a number you can take to a budget meeting.

## Try it yourself

**Find your crossover.** Edit `requests_per_day` and the prices until the monthly bill passes the cost of a dedicated GPU instance, roughly a few hundred dollars a month. That volume is where owning starts to pay. **Stretch:** add a second, larger model's prices and see how much sooner the line crosses.

**Send your real prompt.** Replace the demo prompt with one your product actually uses, and compare the self-hosted latency and token usage against what you expected.

In [ ]:
# Change these, then run the cell.
requests_per_day = 200_000      # your real daily volume
hosted_output_price = 0.60      # your provider's $ per million output tokens

inp = requests_per_day * input_tokens_per_request * 30
out = requests_per_day * output_tokens_per_request * 30
monthly = (inp / 1e6) * hosted_input_price + (out / 1e6) * hosted_output_price
print(f"estimated hosted bill: ${monthly:,.0f} / month")

## Summary

- Inference is the same operation rented or owned. What differs is the path, the price model, and who controls the server.
- The `usage` object is the unit of a hosted bill. On your own GPU those tokens are free at the margin.
- At production volume the hosted bill scales with every token, while a dedicated GPU is a fixed cost. There is a crossover, and you can compute it.
- Owning also buys data residency, rate-limit control, and the freedom to tune the server, which is the rest of this workshop.

## Next

**Module 2: Inference and Memory.** You own the server now, but you cannot yet see inside it. Next you open up a single request and measure where its time goes, time to first token versus inter-token latency, and read the KV cache gauge that explains every bottleneck to come.